# Processed data

Section 5. Reads `output/Tetouan_clean_10min.csv` when the cleaned frame is not already in memory, aggregates each hour, and writes `output/Tetouan_hourly_clean.csv`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
VC_RED = "#D71920"
VC_DARK = "#8B1014"
ZONE_COLORS = ["#D71920", "#4C78A8", "#59A14F"]
zone_cols = ["zone_1_power", "zone_2_power", "zone_3_power"]
weather_cols = [
    "temperature", "humidity", "wind_speed",
    "general_diffuse_flows", "diffuse_flows",
]

if "df" not in globals():
    df = pd.read_csv(Path("output") / "Tetouan_clean_10min.csv", parse_dates=["datetime"])


## 5. Aggregate 10-minute data into hourly data

Total load at each time point of 10 minutes:

$$
\text{total\_power}_t
=
\text{zone\_1}_t + \text{zone\_2}_t + \text{zone\_3}_t
$$

There are 6 observations per hour. The power and weather variables are averaged hourly**.

In [ ]:
df["total_power"] = df[zone_cols].sum(axis=1)

observations_per_hour = (
    df.set_index("datetime")["total_power"]
    .resample("h")
    .size()
)

hour_completeness = observations_per_hour.value_counts().sort_index().to_frame("Hours")
hour_completeness.index.name = "Number of observations per hour"
display(hour_completeness)

assert (observations_per_hour == 6).all(), "Not enough time right 6 observations"

hourly_mean_cols = weather_cols + zone_cols + ["total_power"]

df_hourly = (
    df.set_index("datetime")[hourly_mean_cols]
    .resample("h")
    .mean()
)

df_hourly["observations_per_hour"] = observations_per_hour
df_hourly.index.name = "datetime"

expected_hourly_index = pd.date_range(
    df_hourly.index.min(), df_hourly.index.max(), freq="h"
)
missing_hours = expected_hourly_index.difference(df_hourly.index)

assert len(missing_hours) == 0, "Timestamp missing hourly string"
assert df_hourly.isna().sum().sum() == 0, "Hourly data is missing values"

print(f"Hourly size: {df_hourly.shape[0]:,} hour {df_hourly.shape[1]} post")
print(f"Interview time duration: {df_hourly.index.min()} - {df_hourly.index.max()}")
display(df_hourly.head())

,Hours
Number of observations per hour,
6,8736


Hourly size: 8,736 hours × 10 columns
Period: 2017-01-01 00:00:00 - 2017-12-30 23:00:00


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone_1_power,zone_2_power,zone_3_power,total_power,observations_per_hour
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.20,75.07,0.08,0.06,0.10,"29,197.97","18,026.75","19,252.05","66,476.77",6
2017-01-01 01:00:00,5.55,77.58,0.08,0.06,0.11,"24,657.22","16,078.42","17,042.89","57,778.53",6
2017-01-01 02:00:00,5.05,78.93,0.08,0.06,0.13,"22,083.04","14,330.70","15,676.14","52,089.88",6
2017-01-01 03:00:00,5.00,77.08,0.08,0.06,0.14,"20,811.14","13,219.45","14,883.86","48,914.45",6
2017-01-01 04:00:00,5.10,74.05,0.08,0.06,0.12,"20,475.95","12,921.58","14,317.11","47,714.64",6


## Export the hourly series


In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
hourly_path = output_dir / "Tetouan_hourly_clean.csv"
df_hourly.reset_index().to_csv(hourly_path, index=False)
print(f"Saved {hourly_path}")
